# Grids

To ensure proper spatial coverage while sparing compute time, we sample one pixel per 100km² grid cell to fit the model.

For final predictions, to ensure local specificity, we extract one pixel per 1km².

These grids are used in 7_write_csv to export the final dataframe for analysis.

* **Imports:**
    * Collection 9 MapBiomas Secondary Vegetation Age
    * Collection 9 MapBiomas Land Use Land Cover
    * `categorical`
    * `distance_to_border_mask`
    * `distance_to_secondary_edge`

* **Exports:**
    * `grid_10k_amazon_secondary_edge_removed` to GEE Feature Collection
    * `grid_10k_atlantic_secondary_edge_removed` to GEE Feature Collection
    * `grid_10k_amazon_secondary` to GEE Feature Collection
    * `grid_1k_amazon_secondary` to GEE Feature Collection
    * `grid_1k_amazon_pastureland` to GEE Feature Collection

In [7]:
import ee
import geemap
from utils import *
initialize()

config = ProjectConfig()
roi = config.roi
data_folder = config.data_folder
last_year = config.last_year

## create_grid
Make grid to export one pixel per 10km2 and one pixel per km2.

`cell_size` = resolution in meters to sample

In [8]:
def create_grid(image, region_name = "amazon", cell_size = 10000, file_name = None):
    
    biomes = ee.FeatureCollection('projects/mapbiomas-workspace/AUXILIAR/biomas-2019')

    amazon_geom = (biomes
                .filter(ee.Filter.eq('Bioma', 'Amazônia'))
                .geometry())

    biomes = ee.Image(f"{config.data_folder}/categorical").select("biome")
    
    if region_name == "amazon":
        biome_index = 1
    elif region_name == "atlantic":
        biome_index = 4
    
    biomes = biomes.eq(biome_index).selfMask()

    pixels_to_sample = biomes.reduceResolution(
        ee.Reducer.first(), maxPixels=65536
        ).reproject(
        crs = image.projection().getInfo()['crs'],
        crsTransform = image.projection().getInfo()['transform']
        ).updateMask(image)
    
    image_scale = round(pixels_to_sample.projection().nominalScale().getInfo())
    closest_multiple = round(cell_size / image_scale) * image_scale

    # First, sample locations based only on the age band
    grid = geemap.create_grid(pixels_to_sample.geometry(), closest_multiple, pixels_to_sample.projection())

    grid = grid.filterBounds(amazon_geom.bounds())
    
    # Function to sample one point per valid cell
    def sample_cell(cell):
        sampled_fc = pixels_to_sample.stratifiedSample(
            numPoints = 1,
            classBand = 'biome',
            region = cell.geometry(),
            scale = pixels_to_sample.projection().nominalScale(),
            geometries = True,
            dropNulls = True,
            tileScale = 3
        )

        # Only return a feature if we found one
        return ee.Feature(ee.Algorithms.If(
            sampled_fc.size().gt(0),
            sampled_fc.first(),
            # Return a placeholder that we can filter out later
            ee.Feature(ee.Geometry.Point([0, 0])).set('is_null', True)
        ))

    samples = grid.map(sample_cell)

    # Filter out placeholder features before exporting
    samples = samples.filter(ee.Filter.notEquals('is_null', True))

    if file_name is None:
        return samples
    else:
        export_name = f"grid_{cell_size//1000}k_{region_name}_{file_name}"

        export_task = ee.batch.Export.table.toAsset(
            collection = samples,
            description = export_name,
            assetId = f"{config.data_folder}/{export_name}"
        )
        export_task.start()



In [13]:
distance_to_border_mask = ee.Image(f"{data_folder}/distance_to_border_mask")
age = ee.Image("projects/mapbiomas-public/assets/brazil/lulc/collection9/mapbiomas_collection90_secondary_vegetation_age_v1") \
        .select("secondary_vegetation_age_2020")\
        .updateMask(distance_to_border_mask).rename("age")

edge = ee.Image(f"{data_folder}/distance_to_secondary_edge").gt(30) # non-edge pixel mask (only those surrounded by secondary forests on all sides)
age_edge_removed = age.updateMask(edge)

pastureland = (ee.Image("projects/mapbiomas-public/assets/brazil/lulc/collection9/mapbiomas_collection90_integration_v1")
            .select([f"classification_{year}" for year in config.range_1985_2020])
            .byte()
            .rename([str(year) for year in config.range_1985_2020]))
pastureland = pastureland.select("2020").eq(15).unmask(0).rename("pastureland")

In [11]:
# create_grid(age_edge_removed, region_name = "amazon", cell_size = 10000, file_name = "secondary_edge_removed")
# create_grid(age_edge_removed, region_name = "atlantic", cell_size = 10000, file_name = "secondary_edge_removed")

# create_grid(age, region_name = "amazon", cell_size = 10000, file_name = "secondary")
# create_grid(age, region_name = "amazon", cell_size = 1000, file_name = "secondary")
# create_grid(pastureland, region_name = "amazon", cell_size = 1000, file_name = "pastureland")